## Importing all the needed libraries

In [1]:
!pip install scikit-plot
!pip install scipy==1.11.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 7.6 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [54 lines of output]
      + meson setup /private/var/folders/pw/y3ztzc9n5ns3pn2xk2j59d2w0000gp/T/pip-install-bsuf305w/scipy_07ea1f25a4204bcd8a81538a7a2135ad /private/var/folders/pw/y3ztzc9n5ns3pn2xk2j59d2w0000gp/T/pip-install-bsuf305w/scipy_07ea1f25a4204bcd8a81538a7a2135ad/.mesonpy-5mqvpc3a -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=/private/var/folders/pw/y3ztzc9n5ns3pn2xk2j59d2w0000gp/T/pip-install-bsuf305w/scipy_07ea1f25a4204bcd8a81538a7a2135ad/.mesonpy-5mqvpc3a/meson-python-native-file.ini
      The Meson build system
      Version: 1.10.0
      Source dir: /priva

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [5]:
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score,
)
#from scikitplot.metrics import plot_roc, plot_precision_recall


In [11]:
#importing the dataset_cleaned
data = pd.read_csv('dataset/DM1_game_dataset_cleaned.csv')
#printing all the columns of the dataset
print(data.columns)
data["Rating"].dtype
data["Rating"].head(10)



Index(['Name', 'YearPublished', 'GameWeight', 'MinPlayers', 'MaxPlayers',
       'MfgPlaytime', 'MfgAgeRec', 'NumAlternates', 'NumExpansions',
       'NumImplementations', 'IsReimplementation', 'Kickstarted', 'Rating',
       'HasFamily', 'Category', 'Rank', 'EngagementLevel'],
      dtype='object')


0       Low
1    Medium
2      High
3       Low
4    Medium
5       Low
6      High
7       Low
8       Low
9    Medium
Name: Rating, dtype: object

In [16]:
#setting the x and y variables
X_df = data.drop(columns=["Rating", "Name", "Rank"])
y = data["Rating"].values

In [17]:
#deciding which columns to use as features
feature_summary = []

for col in X_df.columns:
    s = X_df[col]
    
    feature_summary.append({
        "feature": col,
        "dtype": s.dtype,
        "missing_rate": s.isna().mean(),
        "n_unique": s.nunique(),
        "dominant_value_share": s.value_counts(normalize=True).iloc[0]
                                if s.nunique() > 0 else np.nan
    })

feature_summary = pd.DataFrame(feature_summary).sort_values("feature")
feature_summary

,feature,dtype,missing_rate,n_unique,dominant_value_share
12,Category,object,0.0,9,0.511129
13,EngagementLevel,float64,0.0,3514,0.006856
1,GameWeight,float64,0.0,3886,0.115362
11,HasFamily,int64,0.0,2,0.697335
9,IsReimplementation,int64,0.0,2,0.884364
10,Kickstarted,int64,0.0,2,0.847388
3,MaxPlayers,int64,0.0,53,0.321358
5,MfgAgeRec,int64,0.0,20,0.251063
4,MfgPlaytime,int64,0.0,120,0.176151
2,MinPlayers,int64,0.0,10,0.687006


In [18]:
FINAL_FEATURES = [
    'YearPublished',
    'GameWeight',
    'MinPlayers',
    'MaxPlayers',
    'MfgPlaytime',
    'MfgAgeRec',
    'NumAlternates',
    'NumExpansions',
    'NumImplementations',
    'IsReimplementation',
    'Kickstarted',
    'HasFamily',
    'Category',
    'EngagementLevel'
]

X_df = X_df[FINAL_FEATURES]



In [19]:
#easy sanity check 
print(X_df.dtypes)
print(np.unique(y, return_counts=True))


YearPublished           int64
GameWeight            float64
MinPlayers              int64
MaxPlayers              int64
MfgPlaytime             int64
MfgAgeRec               int64
NumAlternates         float64
NumExpansions         float64
NumImplementations    float64
IsReimplementation      int64
Kickstarted             int64
HasFamily               int64
Category               object
EngagementLevel       float64
dtype: object
(array(['High', 'Low', 'Medium'], dtype=object), array([5000, 7241, 9638]))


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# individua automaticamente le colonne
num_features = X_train.select_dtypes(include="number").columns
cat_features = X_train.select_dtypes(exclude="number").columns  

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ],
    remainder="drop"
)


In [23]:
X_train_pre = preprocess.fit_transform(X_train)  # fit SOLO sul train
X_test_pre  = preprocess.transform(X_test)       # transform sul test

print("X_train_pre shape:", X_train_pre.shape)
print("X_test_pre shape:", X_test_pre.shape)


X_train_pre shape: (17503, 22)
X_test_pre shape: (4376, 22)


In [24]:
# ================================
# KNN (base) + Evaluation
# ================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# modello base (scelta ragionevole: k=11, pesi distance per class imbalance)
knn_base = KNeighborsClassifier(n_neighbors=11, weights="distance", p=2)

knn_base.fit(X_train_pre, y_train)

y_pred_base = knn_base.predict(X_test_pre)

print("BASE MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred_base))
print("F1 macro:", f1_score(y_test, y_pred_base, average="macro"))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_base))
print("\nClassification report:\n", classification_report(y_test, y_pred_base))


# ================================
# KNN + GridSearchCV (tuning)
# ================================
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_neighbors": list(range(3, 41, 2)),
    "weights": ["uniform", "distance"],
    "p": [1, 2]  # 1=Manhattan, 2=Euclidean
}

grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_pre, y_train)

best_knn = grid.best_estimator_
print("\nBEST MODEL (CV)")
print("Best params:", grid.best_params_)
print("Best CV F1 macro:", grid.best_score_)

y_pred_best = best_knn.predict(X_test_pre)

print("\nTEST SET PERFORMANCE (Best KNN)")
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("F1 macro:", f1_score(y_test, y_pred_best, average="macro"))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_best))
print("\nClassification report:\n", classification_report(y_test, y_pred_best))


BASE MODEL
Accuracy: 0.57518281535649
F1 macro: 0.5721054444311201

Confusion matrix:
 [[ 496   68  436]
 [  56  856  536]
 [ 258  505 1165]]

Classification report:
               precision    recall  f1-score   support

        High       0.61      0.50      0.55      1000
         Low       0.60      0.59      0.60      1448
      Medium       0.55      0.60      0.57      1928

    accuracy                           0.58      4376
   macro avg       0.59      0.56      0.57      4376
weighted avg       0.58      0.58      0.57      4376

Fitting 5 folds for each of 76 candidates, totalling 380 fits


/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 380, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call



BEST MODEL (CV)
Best params: {'n_neighbors': 33, 'p': 1, 'weights': 'distance'}
Best CV F1 macro: 0.5961808352372183

TEST SET PERFORMANCE (Best KNN)
Accuracy: 0.5982632541133455
F1 macro: 0.5923518059028797

Confusion matrix:
 [[ 479   62  459]
 [  35  890  523]
 [ 207  472 1249]]

Classification report:
               precision    recall  f1-score   support

        High       0.66      0.48      0.56      1000
         Low       0.62      0.61      0.62      1448
      Medium       0.56      0.65      0.60      1928

    accuracy                           0.60      4376
   macro avg       0.62      0.58      0.59      4376
weighted avg       0.61      0.60      0.60      4376



Exception ignored in: <function ResourceTracker.__del__ at 0x107009da0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106d0dda0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x104e65da0>
Traceback (most recent call last

In [ ]:
#